<a href="https://colab.research.google.com/github/rudra629/ml-internship-flyrank/blob/main/work/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import duckdb
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score
import getpass

# 1. Prompt for token
hf_token = getpass.getpass("Enter your Hugging Face read token: ")

# 2. Authenticate DuckDB
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

# 3. Query the data using the exact column names from the error
REL = 'hf://datasets/FlyRank/internship-warehouse'
query = f"""
SELECT
    content_hash_id,
    SUM(gsc_clicks) as total_clicks,
    SUM(gsc_impressions) as total_impressions,
    AVG(gsc_avg_position) as avg_position,
    (SUM(gsc_clicks)*1.0 / NULLIF(SUM(gsc_impressions), 0)) as ctr
FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')
GROUP BY content_hash_id
HAVING total_impressions > 1000
"""
df = con.sql(query).df().dropna()

# 4. Create a label and train the model
ctr_threshold = df['ctr'].median()
df['needs_refresh'] = ((df['total_impressions'] > df['total_impressions'].median()) &
                       (df['ctr'] < ctr_threshold)).astype(int)

X = df[['total_impressions', 'avg_position', 'ctr']]
y = df['needs_refresh']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

baseline_preds = (X_test['avg_position'] > 10).astype(int)

model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)
model_preds = model.predict(X_test)

print(f"Baseline Precision: {precision_score(y_test, baseline_preds):.2f}")
print(f"Model Precision: {precision_score(y_test, model_preds):.2f}")

Enter your Hugging Face read token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Baseline Precision: 0.23
Model Precision: 1.00
